# BÁO CÁO MILESTONE 2 — HIERARCHY NORMALIZER (LÀM SẠCH & CHUẨN HÓA CÂY PHÂN CẤP)
**Nhiệm vụ:** Làm sạch cú pháp LaTeX (*LaTeX cleanup*) và chuẩn hóa công thức toán (*math normalization*) cho nội dung của **mọi nút** trong cây phân cấp `hierarchy_tree*.json` của từng version, đồng thời **giữ nguyên cấu trúc cây và các định danh (ID)** để bước khử trùng lặp phía sau có thể so khớp nội dung ổn định.
**Nhóm:** Nhóm 11 — Nhập môn Khoa học Dữ liệu

---

## 1. Bối Cảnh & Yêu Cầu Liên Quan Trong `parsing.md`

Mục *Standardization and Deduplication → Cleanup and Normalization* của đề bài yêu cầu:

- **LaTeX Cleanup:** loại bỏ các lệnh định dạng không đóng góp ngữ nghĩa, ví dụ `\centering`, tùy chọn vị trí `[htpb]`, `\midrule`.
- **Math Normalization:** đưa toàn bộ toán trong dòng (*inline*) về một dạng thống nhất như `$...$`, và toàn bộ toán khối (*block*) về một dạng thống nhất như `\begin{equation}...\end{equation}`.
- **Thứ tự xử lý:** phần *Deduplication* quy định việc so khớp toàn văn phải thực hiện **sau khi cleanup**, để những khác biệt nhỏ về định dạng hoặc khoảng trắng không làm hai nội dung giống nhau bị coi là khác nhau.

Bảng ánh xạ yêu cầu → thành phần cài đặt:

| Yêu cầu | Thành phần thực hiện | Vị trí |
| :--- | :--- | :--- |
| Duyệt toàn bộ cây, giữ nguyên cấu trúc và ID | `HierarchyNormalizer.normalize_tree` + `_normalize_node` | [`src/parser/hierarchy_normalizer.py`](../src/parser/hierarchy_normalizer.py) |
| Math inline về dạng thống nhất | `HierarchyExtractor.standardize_math` (được gọi trong `normalize_content`) | `src/parser/extractor.py` |
| Math block về `\begin{equation}...\end{equation}` | `HierarchyNormalizer.BLOCK_MATH_PATTERN` (chỉ áp dụng cho nút `kind = "block_formula"`) | `hierarchy_normalizer.py` |
| Bỏ `\centering`, `[htpb]`, `\midrule`, ... | `HierarchyExtractor.clean_latex` (được gọi cuối `normalize_content`) | `src/parser/extractor.py` |
| Chạy hàng loạt trên nhiều publication/version | `normalize_path`, `main` (CLI) và `src/run_normalizer.py` | `hierarchy_normalizer.py`, `run_normalizer.py` |

> **Note:** File `hierarchy_normalizer.py` chỉ chứa phần **điều phối** (duyệt cây, kiểm tra dữ liệu, đọc/ghi file) và biểu thức chính quy xử lý toán khối. Hai hàm `standardize_math` và `clean_latex` thuộc lớp `HierarchyExtractor` trong `extractor.py` và **chỉ được gọi lại** ở đây, không được định nghĩa trong file này. Báo cáo này vì vậy chỉ mô tả *vai trò và vị trí gọi* của hai hàm đó; chi tiết quy tắc thay thế bên trong nằm ở báo cáo của `extractor.py`.

## 2. Luồng Hoạt Động & Luồng Dữ Liệu (Data & Execution Workflow)

Normalizer nhận các file cây phân cấp (một file cho mỗi version của mỗi publication) do bước dựng cây sinh ra, chuẩn hóa `content` của từng nút và ghi ra file `*.normalized.json`. Bộ dữ liệu của nhóm có thể được truyền vào cả dạng **thư mục** (xử lý hàng loạt) lẫn **một file** (xử lý đơn lẻ).

```text
[Đầu vào]
   └── <source_dir>/<arxiv_id>/vX/hierarchy_tree.json      (cây phân cấp của từng version; mỗi nút có kind, content, children, ...)
          │
          ▼
[Bước Normalize (src/parser/hierarchy_normalizer.py)]
   ├── Giai đoạn 1: Tìm file & tính đường dẫn đầu ra  (normalize_path)
   │      ├── Đầu vào là THƯ MỤC: rglob("hierarchy_tree*.json"), bỏ qua file có stem kết thúc bằng ".normalized"
   │      ├── Đầu vào là FILE: bắt buộc có đuôi .json
   │      ├── Thư mục đầu ra: mặc định <tên_thư_mục>_normalized (cùng cấp) hoặc cùng thư mục với file đầu vào
   │      └── Tên file đầu ra: <stem>.normalized.json; báo lỗi nếu đã tồn tại và không có --overwrite
   │
   ├── Giai đoạn 2: Đọc & kiểm tra cấu trúc  (normalize_file → normalize_tree)
   │      ├── Đọc JSON UTF-8; lỗi đọc file hoặc lỗi cú pháp JSON → ValueError
   │      ├── Kiểm tra tree là object JSON và có khóa "root"
   │      └── Tạo bản sao sâu (copy.deepcopy): dữ liệu gốc không bị sửa
   │
   ├── Giai đoạn 3: Chuẩn hóa từng nút  (_normalize_node: đệ quy theo chiều sâu trên bản sao)
   │      ├── Kiểm tra: nút là object, không có chu trình, content là chuỗi, children là danh sách
   │      └── normalize_content(content, kind):
   │             1) HierarchyExtractor.standardize_math   → thống nhất ký hiệu bao công thức toán
   │             2) nếu kind == "block_formula": BLOCK_MATH_PATTERN → \begin{equation}...\end{equation}
   │             3) HierarchyExtractor.clean_latex        → bỏ lệnh trình bày không mang ngữ nghĩa
   │
   └── Giai đoạn 4: Ghi kết quả  (json.dumps, ensure_ascii=False, indent=2, thêm "\n" cuối file)
          │
          ▼
[Đầu ra]
   └── <thư mục đầu ra>/<...>/hierarchy_tree.normalized.json   (cùng cấu trúc, cùng ID, content đã chuẩn hóa)
```

> **Note:** Hai thứ **không** bị thay đổi trong suốt quá trình: (1) hình dạng cây (số nút, thứ tự và quan hệ cha–con) và (2) mọi trường khác ngoài `content` (ví dụ `element_id`, `kind`). Nhờ đó, sau normalize, các bước phía sau (khử trùng lặp toàn văn, dựng bảng `elements` và `hierarchy`) vẫn dùng được cùng bộ định danh nút. Việc làm sạch **trước** khi so khớp toàn văn là điều kiện để hai câu chỉ khác nhau về khoảng trắng hay lệnh định dạng được nhận ra là một.

## 3. Các Thư Viện & Công Cụ Sử Dụng

| Thư viện / Module | Chức năng trong phần Hierarchy Normalizer |
| :--- | :--- |
| `re` (Regular Expressions) | Biên dịch `BLOCK_MATH_PATTERN` để tìm các môi trường toán khối (`equation`, `align`, `gather`, ...) kèm thân công thức, dùng nhóm có tên `env`/`body` và *backreference* `(?P=env)` để đảm bảo thẻ mở và thẻ đóng cùng loại. |
| `json` | Đọc (`json.loads`) và ghi (`json.dumps` với `ensure_ascii=False`, `indent=2`) file cây phân cấp, giữ nguyên ký tự Unicode (chữ Việt, ký hiệu toán). |
| `pathlib.Path` | Duyệt thư mục (`rglob`), tính đường dẫn tương đối (`relative_to`), tạo thư mục đầu ra (`mkdir(parents=True, exist_ok=True)`), đọc/ghi văn bản. |
| `copy` | `copy.deepcopy` tạo bản sao độc lập của cây để chuẩn hóa tại chỗ mà không làm thay đổi dữ liệu đầu vào. |
| `argparse` | Giao diện dòng lệnh: `--input`, `--output-dir`, `--overwrite` (trong `hierarchy_normalizer.py`) và `--source-dir` (trong `run_normalizer.py`). |
| `typing` (`Any`, `Dict`, `List`, `Optional`, `Set`) | Chú thích kiểu cho tham số và giá trị trả về. |
| `from __future__ import annotations` | Cho phép dùng chú thích kiểu dạng chuỗi (hoãn đánh giá), ví dụ `List[str] \| None` trong `run_normalizer.py`. |
| `parser.extractor.HierarchyExtractor` | Cung cấp `standardize_math` (chuẩn hóa ký hiệu toán) và `clean_latex` (loại lệnh LaTeX không mang ngữ nghĩa). Được import bằng *relative import* `from .extractor import HierarchyExtractor`. |
| `parser.pipeline.ParsingPipeline` | Được `run_normalizer.py` dùng qua `find_hierarchy_trees` và `normalize_source` để tìm và chuẩn hóa các cây theo từng version. |
| `src/parser/hierarchy_normalizer.py` | Module lõi: lớp `HierarchyNormalizer`, các hàm `_is_relative_to`, `normalize_path`, `main`. |
| `src/run_normalizer.py` | Script CLI chạy chuẩn hóa cho toàn bộ thư mục dữ liệu của dự án (mặc định `sourcesArxiv`). |

> **Note:** Vì dùng *relative import* (`from .extractor import ...`), module phải được nạp như một phần của package `parser` (ví dụ `from parser.hierarchy_normalizer import ...` hoặc `python -m parser.hierarchy_normalizer ...` chạy từ thư mục `src`). Chạy trực tiếp `python src/parser/hierarchy_normalizer.py` sẽ gây lỗi import.

## 4. Giải Thích Chi Tiết Mã Nguồn

### 4.1. Dữ liệu đầu vào và đầu ra: cấu trúc file cây phân cấp

Normalizer làm việc với một **object JSON** có khóa bắt buộc `root`. Mỗi nút là một object lồng nhau:

```json
{
  "root": {
    "element_id": "2510.10788#root1",
    "kind": "document",
    "content": "Version 1 root",
    "children": [
      {"element_id": "2510.10788#sec1", "kind": "section", "content": "Introduction", "children": [
        {"element_id": "2510.10788#s1", "kind": "sentence", "content": "Let \\(x\\) be a variable.", "children": []}
      ]}
    ]
  }
}
```

Các trường mà code **đọc hoặc kiểm tra**:

| Trường | Vai trò trong normalizer | Quy tắc |
| :--- | :--- | :--- |
| `root` (cấp cao nhất) | Nút gốc của cây | Bắt buộc phải có khóa `root`, nếu thiếu → `ValueError`. Giá trị `null` được chấp nhận: bản sao được trả về nguyên vẹn, không xử lý nút nào. |
| `content` | Văn bản cần chuẩn hóa | Nếu `None` hoặc không có: bỏ qua. Nếu có thì **phải là chuỗi**, ngược lại → `ValueError`. |
| `kind` | Quyết định có áp dụng `BLOCK_MATH_PATTERN` hay không | Chỉ giá trị `"block_formula"` kích hoạt bước đổi môi trường toán khối. |
| `children` | Danh sách nút con để đệ quy | Nếu `None` hoặc không có: nút lá. Nếu có thì **phải là danh sách**, ngược lại → `ValueError`. |
| Các trường khác (`element_id`, ...) | Không đọc, không sửa | Được giữ nguyên nhờ `deepcopy`. |

**Đầu ra** là một object JSON cùng cấu trúc và cùng các trường, chỉ có giá trị `content` của các nút thay đổi, được ghi với `indent=2`, `ensure_ascii=False` và một ký tự xuống dòng ở cuối file.

---

### 4.2. Lớp `HierarchyNormalizer` trong [`src/parser/hierarchy_normalizer.py`](../src/parser/hierarchy_normalizer.py)

Lớp chỉ chứa **hằng số lớp và `@classmethod`** (không cần khởi tạo đối tượng, không giữ trạng thái giữa các lần gọi). Docstring của lớp: *"Normalize node content while preserving the tree structure and IDs."*

**a) Hằng số lớp**

- `BLOCK_MATH_ENVIRONMENTS`: chuỗi regex liệt kê 7 họ môi trường toán khối: `equation`, `align`, `gather`, `multline`, `eqnarray`, `flalign` (mỗi loại có thể có dấu `*`) và `displaymath`.
- `BLOCK_MATH_PATTERN`: regex đã biên dịch với cờ `re.DOTALL`, dạng

  `\begin{(?P<env>...)}(?P<body>.*?)\end{(?P=env)}`

  - nhóm `env` bắt tên môi trường; `(?P=env)` buộc thẻ đóng **trùng** tên với thẻ mở (ví dụ `align*` chỉ khớp với `align*`);
  - nhóm `body` bắt toàn bộ thân công thức, `.*?` là *non-greedy* để dừng ở thẻ đóng đầu tiên phù hợp, `re.DOTALL` cho phép thân công thức trải nhiều dòng.

**b) Phương thức `normalize_content(content, kind=None) -> str`** — chuẩn hóa nội dung của **một** nút

1. Gọi `HierarchyExtractor.standardize_math(content)` để thống nhất cách bao công thức toán.
2. Nếu `kind == "block_formula"`: dùng `BLOCK_MATH_PATTERN.sub(...)` thay mỗi môi trường tìm được bằng
   `\begin{equation}` + xuống dòng + `body.strip()` + xuống dòng + `\end{equation}`.
   Với các nút có `kind` khác, bước này **bị bỏ qua**.
3. Gọi `HierarchyExtractor.clean_latex(...)` trên kết quả để loại bỏ lệnh định dạng không mang ngữ nghĩa, rồi trả về chuỗi cuối cùng.

> **Note:** Thứ tự ba bước là có chủ đích: chuẩn hóa ký hiệu toán trước, đổi môi trường toán khối ở giữa, và dọn dẹp lệnh trình bày sau cùng trên chuỗi đã hoàn chỉnh. Pattern chỉ bắt các môi trường dạng `\begin{...}\end{...}`; các dạng khác (ví dụ `$$...$$`) không do pattern này xử lý, nên việc chuyển sang `equation` cho các dạng đó (nếu có) phụ thuộc vào `standardize_math`.

**c) Phương thức `normalize_tree(tree) -> Dict`** — chuẩn hóa **một cây** đã nạp vào bộ nhớ

1. Kiểm tra `tree` là `dict` (nếu không → `ValueError("Hierarchy tree must be a JSON object.")`) và có khóa `root` (nếu không → `ValueError("... missing its 'root' field.")`).
2. `copy.deepcopy(tree)` tạo bản sao: **đối tượng truyền vào không bị thay đổi**.
3. Nếu `root` khác `None`, gọi `_normalize_node(root, set())` để chuẩn hóa toàn bộ cây trên bản sao.
4. Trả về bản sao đã chuẩn hóa.

**d) Phương thức `_normalize_node(node, active_nodes) -> None`** — duyệt đệ quy theo chiều sâu (DFS)

- Kiểm tra `node` là `dict`.
- **Phát hiện chu trình:** `active_nodes` là tập `id(node)` của các nút **đang nằm trên đường đi hiện tại** (thêm vào trước khi xử lý, gỡ ra trong khối `finally`). Gặp lại một nút đang ở trong tập → `ValueError("Hierarchy tree contains a cycle.")`. Vì gỡ ra khi thoát nút, cùng một đối tượng xuất hiện ở hai nhánh khác nhau không bị báo nhầm là chu trình.
- **Xử lý nội dung:** nếu `content` không phải `None` thì kiểm tra kiểu chuỗi rồi **gán lại** `node["content"] = normalize_content(content, kind=node.get("kind"))`.
- **Xử lý nút con:** nếu `children` không phải `None` thì kiểm tra là danh sách rồi gọi đệ quy cho từng nút con.
- Hàm sửa trực tiếp trên bản sao (không trả về giá trị).

**e) Phương thức `normalize_file(input_path, output_path) -> Path`** — chuẩn hóa **một file**

1. Đọc `input_path` bằng UTF-8 và `json.loads`; mọi `OSError` hoặc `JSONDecodeError` được bọc thành `ValueError("Unable to read hierarchy tree '...': ...")`.
2. Gọi `normalize_tree`.
3. Tạo thư mục cha của `output_path` nếu chưa có, ghi `json.dumps(normalized, ensure_ascii=False, indent=2) + "\n"`.
4. Trả về `output_path`. Docstring cho phép `output_path` **trùng** `input_path` (ghi đè tại chỗ), do toàn bộ file đã được đọc xong trước khi ghi.

---

### 4.3. Các hàm cấp module trong `hierarchy_normalizer.py`

**a) `_is_relative_to(path, parent) -> bool`** — thay cho `Path.is_relative_to` (chỉ có từ Python 3.9): thử `path.relative_to(parent)`, thành công → `True`, bắt `ValueError` → `False`.

**b) `normalize_path(input_path, output_dir=None, overwrite=False) -> List[Path]`** — hàm điều phối chính

1. `resolve()` đường dẫn đầu vào; không tồn tại → `FileNotFoundError`.
2. Chia hai nhánh theo loại đầu vào:

| | Đầu vào là **thư mục** | Đầu vào là **một file** |
| :--- | :--- | :--- |
| Danh sách file xử lý | `sorted(rglob("hierarchy_tree*.json"))`, loại file có `stem` kết thúc bằng `.normalized`; rỗng → `FileNotFoundError` | Chính file đó; đuôi phải là `.json`, nếu không → `ValueError` |
| Thư mục đầu ra mặc định | `<thư mục cha của input>/<tên input>_normalized` | Cùng thư mục với file đầu vào |
| Khi có `--output-dir` | Dùng thư mục đó, nhưng **không được nằm trong** thư mục đầu vào, nếu không → `ValueError` (tránh trộn file sinh ra với cây nguồn và bị `rglob` quét lại) | Dùng thư mục đó (không giữ cấu trúc thư mục con) |
| Vị trí file đầu ra | `output_root / <đường dẫn tương đối của file>.parent / <stem>.normalized.json` (**giữ nguyên cấu trúc thư mục** `<arxiv_id>/vX/`) | `output_root / <stem>.normalized.json` |

3. Với mỗi file: nếu file đầu ra đã tồn tại và `overwrite=False` → `FileExistsError` (kèm gợi ý dùng `--overwrite`); ngược lại gọi `HierarchyNormalizer.normalize_file` và gom đường dẫn đầu ra.
4. Trả về danh sách đường dẫn đầu ra, theo thứ tự đã sắp xếp.

**c) `main(argv=None) -> int`** — giao diện dòng lệnh

| Tham số | Bắt buộc | Ý nghĩa |
| :--- | :--- | :--- |
| `--input` | Có | File `hierarchy_tree*.json` hoặc thư mục chứa các file đó |
| `--output-dir` | Không | Thư mục đầu ra (mặc định như bảng ở 4.3.b) |
| `--overwrite` | Không | Cho phép thay thế file `*.normalized.json` đã có |

Sau khi chạy, `main` in `Normalized N hierarchy tree(s).` và liệt kê từng đường dẫn đầu ra, trả về mã `0`. Ví dụ:

```bash
cd src
python -m parser.hierarchy_normalizer --input sourcesArxiv --output-dir sourcesArxiv_normalized --overwrite
```

---

### 4.4. Script điều phối `run_normalizer.py` trong [`src/run_normalizer.py`](../src/run_normalizer.py)

Script này là **cổng chạy cho toàn pipeline**, tách biệt với CLI của `hierarchy_normalizer.py`:

- Thêm thư mục chứa script vào `sys.path` rồi import `ParsingPipeline` từ `parser.pipeline`.
- `find_hierarchy_trees(source_dir)`: hàm bao (*compatibility wrapper*) gọi `ParsingPipeline.find_hierarchy_trees` để tìm các cây sinh ra cho từng version.
- `normalize_publications(source_dir)`: kiểm tra thư mục tồn tại (`FileNotFoundError` nếu không), gọi `ParsingPipeline.normalize_source(source_dir)` để chuẩn hóa **từng cây theo từng version** (docstring ghi *"in place"*), nếu không tìm thấy cây nào → `FileNotFoundError`.
- `main`: tham số `--source-dir`, mặc định `src/sourcesArxiv`, nếu thư mục không tồn tại thì dùng `<thư mục cha>/source`. In `Normalized N hierarchy tree(s).` và danh sách file đầu ra.

> **Note:** Phần bên trong của `ParsingPipeline` (cách tìm cây và cách gọi `HierarchyNormalizer`) không nằm trong các file được mô tả ở báo cáo này.

---

### 4.5. Tổng hợp các lỗi được chủ động kiểm soát

| Tình huống | Ngoại lệ | Nơi phát sinh |
| :--- | :--- | :--- |
| Đường dẫn đầu vào không tồn tại | `FileNotFoundError` | `normalize_path`; `normalize_publications` |
| Thư mục không có file `hierarchy_tree*.json` | `FileNotFoundError` | `normalize_path`; `normalize_publications` |
| Đầu vào là file nhưng không có đuôi `.json` | `ValueError` | `normalize_path` |
| Thư mục đầu ra nằm trong thư mục đầu vào | `ValueError` | `normalize_path` |
| File đầu ra đã tồn tại, không có `--overwrite` | `FileExistsError` | `normalize_path` |
| Không đọc được file hoặc JSON sai cú pháp | `ValueError` | `normalize_file` |
| Cây không phải object hoặc thiếu `root` | `ValueError` | `normalize_tree` |
| Nút không phải object, `content` không phải chuỗi, `children` không phải danh sách, có chu trình | `ValueError` | `_normalize_node` |

## 5. Minh Họa Trực Tiếp Bằng Cách Import Code Từ `hierarchy_normalizer`

Các ô dưới đây **import và chạy trực tiếp** lớp `HierarchyNormalizer` và hàm `normalize_path` từ `src/parser/hierarchy_normalizer.py` trên dữ liệu mẫu nhỏ. Kết quả cụ thể của bước thống nhất ký hiệu toán và dọn lệnh LaTeX do `HierarchyExtractor` quyết định, nên các bảng bên dưới phản ánh đúng phiên bản `extractor.py` đang có trong repo.

In [ ]:
import re
import sys
import json
import tempfile
import pandas as pd
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "reports" else Path.cwd()
src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from parser.hierarchy_normalizer import HierarchyNormalizer, normalize_path

print("Các môi trường toán khối được nhận diện:")
print(" ", HierarchyNormalizer.BLOCK_MATH_ENVIRONMENTS.replace("\\*?", "[*]").replace("|", ", "))
print("\nBiểu thức chính quy BLOCK_MATH_PATTERN:")
print(" ", HierarchyNormalizer.BLOCK_MATH_PATTERN.pattern[:80] + " ...")
print("  dùng re.DOTALL:", bool(HierarchyNormalizer.BLOCK_MATH_PATTERN.flags & re.DOTALL))

### 5.1. `normalize_content`: chuẩn hóa nội dung của từng loại nút

Mỗi dòng dưới đây là một nút giả lập (cặp `kind` – `content`). Cột *Trước* là nội dung gốc, cột *Sau* là kết quả của `HierarchyNormalizer.normalize_content(content, kind)`.

In [ ]:
samples = [
    ("sentence",      r"Let \(x\) be a real variable and $y$ a parameter."),
    ("block_formula", "\\begin{align*}\n  a &= b + c \\\\\n  d &= e\n\\end{align*}"),
    ("block_formula", r"\begin{equation}  E = mc^2  \end{equation}"),
    ("block_formula", r"$$ x^2 + y^2 = z^2 $$"),
    ("figure",        r"\centering \includegraphics[width=\linewidth]{fig1.png} \caption{Overview of the model}"),
    ("sentence",      "We   study   the   problem   of   citation   matching."),
]

rows = []
for kind, content in samples:
    rows.append({
        "kind": kind,
        "Trước": content,
        "Sau": HierarchyNormalizer.normalize_content(content, kind=kind),
    })

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
df_content = pd.DataFrame(rows)
print("BẢNG 1: KẾT QUẢ normalize_content TRÊN CÁC LOẠI NÚT")
display(df_content)

print("\nChi tiết nút block_formula đầu tiên (hiển thị nguyên văn, có xuống dòng):")
print("--- Trước ---");  print(samples[1][1])
print("--- Sau ---");    print(HierarchyNormalizer.normalize_content(samples[1][1], kind="block_formula"))

### 5.2. `normalize_tree`: chuẩn hóa cả cây, giữ nguyên cấu trúc và ID

Cây mẫu bên dưới mô phỏng một version có Document → Section → (câu, công thức khối, hình). Sau `normalize_tree`, ta so sánh từng nút trước và sau để thấy **chỉ `content` thay đổi**, còn `element_id`, `kind` và quan hệ cha–con được giữ nguyên, đồng thời cây gốc không bị sửa.

In [ ]:
tree = {
    "root": {
        "element_id": "2510.10788#root1", "kind": "document", "content": "Version 1 root",
        "children": [{
            "element_id": "2510.10788#sec1", "kind": "section", "content": "Introduction",
            "children": [
                {"element_id": "2510.10788#s1", "kind": "sentence", "children": [],
                 "content": r"Let \(x\) be a real variable   and $y$ a parameter."},
                {"element_id": "2510.10788#eq1", "kind": "block_formula", "children": [],
                 "content": "\\begin{align*}\n  a &= b + c \\\\\n  d &= e\n\\end{align*}"},
                {"element_id": "2510.10788#fig1", "kind": "figure", "children": [],
                 "content": r"\centering \includegraphics[width=\linewidth]{fig1.png} \caption{Overview}"},
            ],
        }],
    }
}

def flatten(node, parent=None, out=None):
    out = [] if out is None else out
    out.append({"element_id": node["element_id"], "kind": node["kind"], "parent_id": parent,
                "n_children": len(node.get("children") or []), "content": node.get("content")})
    for child in node.get("children") or []:
        flatten(child, node["element_id"], out)
    return out

before = pd.DataFrame(flatten(tree["root"]))
snapshot = json.dumps(tree, sort_keys=True)

normalized_tree = HierarchyNormalizer.normalize_tree(tree)
after = pd.DataFrame(flatten(normalized_tree["root"]))

compare = before[["element_id", "kind", "parent_id", "n_children"]].copy()
compare["content_trước"] = before["content"]
compare["content_sau"] = after["content"]
print("BẢNG 2: SO SÁNH TỪNG NÚT TRƯỚC & SAU normalize_tree")
display(compare)

same_shape = before[["element_id", "kind", "parent_id", "n_children"]].equals(after[["element_id", "kind", "parent_id", "n_children"]])
print(f"\nSố nút: {len(before)} → {len(after)}")
print(f"Cấu trúc, ID và kind giữ nguyên: {same_shape}")
print(f"Cây đầu vào không bị sửa đổi (deepcopy): {json.dumps(tree, sort_keys=True) == snapshot}")
print(f"Số nút có content thay đổi: {int((before['content'] != after['content']).sum())}")

### 5.3. `normalize_path`: quy trình file → file với thư mục dữ liệu

Ô dưới đây dựng một thư mục tạm có hai version của một publication, chạy `normalize_path` ở chế độ **thư mục**, rồi hiển thị file đầu vào, file đầu ra sinh ra và ví dụ hành vi khi file đầu ra đã tồn tại (`overwrite`). Thư mục tạm bị xóa khi ô chạy xong, không làm ảnh hưởng dữ liệu thật.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    input_root = tmp / "sourcesDemo"
    for version in ("v1", "v2"):
        version_dir = input_root / "2510.10788" / version
        version_dir.mkdir(parents=True)
        (version_dir / "hierarchy_tree.json").write_text(json.dumps(tree, ensure_ascii=False, indent=2), encoding="utf-8")

    print("ĐẦU VÀO:")
    for p in sorted(input_root.rglob("*.json")):
        print("  ", p.relative_to(tmp))

    outputs = normalize_path(input_root)     # thư mục đầu ra mặc định: sourcesDemo_normalized
    print("\nĐẦU RA (mặc định, giữ nguyên cấu trúc thư mục <arxiv_id>/<version>/):")
    for p in outputs:
        print("  ", p.relative_to(tmp))

    print("\nChạy lại khi file đầu ra đã tồn tại, không có overwrite:")
    try:
        normalize_path(input_root)
    except FileExistsError as error:
        print("  FileExistsError:", str(error).replace(str(tmp), "<tmp>"))

    outputs_again = normalize_path(input_root, overwrite=True)
    print(f"\nVới overwrite=True: ghi lại {len(outputs_again)} file.")

    result = json.loads(outputs[0].read_text(encoding="utf-8"))
    print("\nNội dung nút 's1' trong file đầu ra đầu tiên:")
    print("  ", result["root"]["children"][0]["children"][0]["content"])

## 6. Lưu Ý Khi Sử Dụng & Hạn Chế Của Cài Đặt Hiện Tại

1. **Đổi `align`, `gather`, `multline`, `eqnarray`, `flalign` thành `equation`:** thân công thức được giữ nguyên, nhưng các môi trường nhiều dòng thường chứa ký hiệu căn lề `&` và ngắt dòng `\\`. Trong LaTeX, `equation` không hỗ trợ trực tiếp các ký hiệu này (thường phải bọc trong `aligned`/`split`). Dạng có dấu `*` (không đánh số) cũng trở thành `equation` có đánh số. Với mục tiêu của milestone (thống nhất dạng biểu diễn để so khớp văn bản) điều này chấp nhận được, nhưng chuỗi đầu ra không nên được biên dịch lại như nguồn LaTeX gốc.
2. **Chỉ nút `kind = "block_formula"` mới được đổi môi trường:** môi trường toán xuất hiện bên trong nút có `kind` khác (ví dụ trong câu) không bị `BLOCK_MATH_PATTERN` động tới.
3. **Regex *non-greedy*:** thẻ đóng được ghép với thẻ mở gần nhất cùng tên; các môi trường **cùng loại lồng nhau** sẽ không được xử lý chính xác. Trường hợp này hiếm với công thức khối.
4. **Phụ thuộc vào `HierarchyExtractor`:** chất lượng cleanup và thống nhất toán trong dòng hoàn toàn do `standardize_math` và `clean_latex` quyết định; normalizer chỉ gọi chúng theo thứ tự cố định.
5. **Không có tính nguyên tử khi chạy hàng loạt:** `FileExistsError` được kiểm tra ngay trước khi ghi từng file, nên nếu một file đầu ra đã tồn tại ở giữa danh sách, các file đứng trước nó đã được ghi. Chạy lại với `--overwrite` hoặc dùng thư mục đầu ra mới.
6. **Chế độ một file không loại trừ `.normalized`:** việc bỏ qua file đã chuẩn hóa chỉ áp dụng khi quét thư mục; truyền trực tiếp `x.normalized.json` sẽ sinh `x.normalized.normalized.json`.
7. **Mẫu quét `hierarchy_tree*.json`:** khớp cả `hierarchy_tree.json` và các file dạng `hierarchy_tree_vX.json`, nên nếu thư mục đầu vào chứa cả hai, cả hai đều được chuẩn hóa.
8. **Đệ quy:** `_normalize_node` đệ quy theo độ sâu của cây, và giới hạn đệ quy mặc định của Python (1000) có thể bị chạm với cây lồng cực sâu; cây phân cấp tài liệu thường chỉ sâu vài chục mức.

## 7. Tóm Tắt Đầu Vào – Đầu Ra

| Thành phần | Đầu vào | Đầu ra |
| :--- | :--- | :--- |
| `normalize_content(content, kind)` | Chuỗi nội dung của một nút và `kind` của nút | Chuỗi đã chuẩn hóa toán và làm sạch LaTeX |
| `normalize_tree(tree)` | `dict` cây phân cấp có khóa `root` | **Bản sao** cây với `content` đã chuẩn hóa, cấu trúc và ID giữ nguyên |
| `normalize_file(input_path, output_path)` | Đường dẫn file JSON cây và đường dẫn đầu ra | File JSON đã chuẩn hóa (UTF-8, `indent=2`); trả về `output_path` |
| `normalize_path(input_path, output_dir, overwrite)` | File hoặc thư mục chứa `hierarchy_tree*.json` | Danh sách đường dẫn `*.normalized.json` (giữ cấu trúc thư mục khi đầu vào là thư mục) |
| `python -m parser.hierarchy_normalizer` | `--input`, `--output-dir`, `--overwrite` | In số lượng cây đã chuẩn hóa và đường dẫn từng file |
| `src/run_normalizer.py` | `--source-dir` (mặc định `sourcesArxiv`) | Chuẩn hóa cây của từng version qua `ParsingPipeline`, in số lượng và đường dẫn |